# Photorealistic Image Generator — Google Colab

This notebook uses **RealVisXL V4.0** with Hugging Face Diffusers to generate realistic, photography-style images from text prompts. It is intended for a Colab GPU runtime; 24 GB VRAM is a comfortable target, while smaller GPUs may need more CPU offloading and patience.

## Before you run
- In Colab, choose **Runtime → Change runtime type → GPU**.
- Run the cells from top to bottom.
- The first model load downloads several gigabytes and may take a while.
- Read the model card and its **OpenRAIL++** licence before using outputs, especially commercially: https://huggingface.co/SG161222/RealVisXL_V4.0
- Use only lawful prompts. Do not create sexual images of minors or non-consensual intimate imagery of real people. This notebook does not grant permission to use someone's likeness.

The model can generate adult-oriented content, but this notebook does not provide prompts or workflows for depicting real people in intimate situations. You are responsible for following the model licence and applicable law.

## If the setup cell you see still says `diffusers==0.29.2` or tries to build `tokenizers`

You are running an older cached copy. Reopen the latest notebook link. The current setup requires prebuilt binary wheels (`--only-binary=:all:`), so it stops with a clear error rather than falsely printing success after a failed install. Do not continue if setup raises an error.

In [ ]:
# Run this cell first. It refuses to build tokenizers from source; it must install a prebuilt wheel.
%pip uninstall -y torchao
import subprocess, sys
cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade', '--only-binary=:all:', 'diffusers', 'transformers>=4.48.0', 'tokenizers>=0.21.0', 'accelerate', 'safetensors']
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError('Package installation failed. Do not run the remaining cells. In Colab, choose Runtime → Disconnect and delete runtime, reconnect, then reopen the latest notebook and retry this setup cell.')
print('Installation succeeded. Now choose Runtime → Restart session, then run all cells from the top.')

In [ ]:
import torch
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab, select Runtime → Change runtime type → GPU, then rerun this cell."
    )

props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB")
print(f"PyTorch: {torch.__version__}")

## Load the model

CPU offloading reduces GPU memory use, though it can make generation slower. The model weights are downloaded from Hugging Face; they are not stored in this GitHub repository.

In [ ]:
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

MODEL_ID = "SG161222/RealVisXL_V4.0"

pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

# Karras sigmas are a common choice for this style of SDXL generation.
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
)

# Helps fit the model on limited VRAM. Requires accelerate (installed above).
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print("Model loaded. You can now generate an image.")

## Generate an image

Edit the prompt to describe the subject, camera, lens, lighting, composition, and setting. Start with one image at 1024×1024 and 25–30 steps. Detailed prompts help, but extra words such as “8K” do not guarantee better output.

In [ ]:
from pathlib import Path
from IPython.display import display

# Example prompt: a fictional adult subject, not a real person.
prompt = """Photorealistic editorial photograph of a fictional adult woman in her late 20s,
natural skin texture and subtle pores, realistic eyes, authentic facial proportions,
soft window light, 85mm portrait lens, shallow depth of field, neutral colour grading,
candid expression, high-end photography, detailed but natural, lifelike background"""

negative_prompt = """low quality, blurry, oversharpened, plastic skin, waxy skin,
over-smoothed face, unnatural eyes, asymmetrical eyes, malformed hands, extra fingers,
missing fingers, duplicate limbs, distorted anatomy, text, watermark, logo, jpeg artifacts"""

# Supported dimensions should be multiples of 8. Start at 1024 for SDXL.
width = 1024
height = 1024
steps = 28
guidance = 5.5
seed = 123456  # Change this number for a different random result.

if width % 8 or height % 8:
    raise ValueError("Width and height must be multiples of 8.")
if not (512 <= width <= 1024 and 512 <= height <= 1024):
    raise ValueError("For this starter notebook, use dimensions between 512 and 1024.")
if not (1 <= steps <= 60):
    raise ValueError("Steps must be between 1 and 60.")

generator = torch.Generator(device="cuda").manual_seed(seed)

with torch.inference_mode():
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_inference_steps=steps,
        guidance_scale=guidance,
        num_images_per_prompt=1,
        generator=generator,
    )

image = result.images[0]
display(image)

output_path = Path("/content/realvisxl_output.png")
image.save(output_path)
print(f"Saved image to: {output_path}")

In [ ]:
# Optional: download the generated PNG to your device
from google.colab import files
files.download(str(output_path))

## Troubleshooting
- **CUDA out of memory:** Restart the runtime, reduce width/height to 768, and rerun the model-loading and generation cells. CPU offload is already enabled.
- **Model download is slow:** The first download is several gigabytes; wait for it to finish.
- **No GPU:** Change the Colab runtime to GPU. Availability depends on your account and current Colab limits.
- **Commercial use:** Review the current model card and licence before monetizing the notebook or its outputs.

Generated images may contain visual errors. Inspect results before publishing them.